In [1]:
from huggingface_hub import login
login()

## Load model from Hugging Face Hub

In [2]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "4b-it"
model_id = f"google/medgemma-{model_variant}"

use_quantization = False

model_kwargs = dict(
    dtype=torch.bfloat16,
    device_map="auto",
)

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

The following sections contain standalone examples demonstrating how to use the model both directly and with the [`pipeline`](https://huggingface.co/docs/transformers/en/main_classes/pipelines) API. The `pipeline` API provides a simple way to use the model for inference while abstracting away complex details,  while directly using the model gives you complete control over the inference process, including preprocessing and postprocessing. In practice, you should select the method that is best suited for your use case.

Here, you will load the model directly and with the `pipeline` API for use in the next sections. Note that the multimodal variants and the 27B text-only variant are loaded with their respective tasks and classes.

**Load model with the `pipeline` API**

In [3]:
from transformers import pipeline

if "text" in model_variant:
    pipe = pipeline("text-generation", model=model_id, model_kwargs=model_kwargs)
else:
    pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

pipe.model.generation_config.do_sample = False

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda:0


In [6]:
from IPython.display import Markdown

prompt = """

به پرسش زیر قدم به قدم فکر کنید و زنجیره افکار (chain of thought) خود برای رسیدن به پاسخ را به طور کامل شرح دهید و تنها یک گزینه را انتخاب کنید.
## پرسش
آنزیم های گلیکولیز در موارد زیر قرار دارند:


## گزینه ها
[A] : میتوکندری
[B] : هسته
[C] : سیتوپلاسم
[D] : لیزوزوم ها


"""

role_instruction = "You are a helpful medical assistant."

system_instruction = role_instruction
max_new_tokens = 500

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }
]

**Run model with the `pipeline` API**

In [7]:
output = pipe(messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**



به پرسش زیر قدم به قدم فکر کنید و زنجیره افکار (chain of thought) خود برای رسیدن به پاسخ را به طور کامل شرح دهید و تنها یک گزینه را انتخاب کنید.
## پرسش
آنزیم های گلیکولیز در موارد زیر قرار دارند: 


## گزینه ها
[A] : میتوکندری
[B] : هسته
[C] : سیتوپلاسم
[D] : لیزوزوم ها 




---

**[ MedGemma ]**

به عنوان یک دستیار پزشکی، من باید زنجیره افکار خود را برای رسیدن به پاسخ به این سوال شرح دهم.

1.  **گلیکولیز چیست؟** گلیکولیز یک فرآیند متابولیک است که در سلول ها رخ می دهد و گلوکز را به پیروات تبدیل می کند. این فرآیند در سیتوپلاسم سلول ها اتفاق می افتد.
2.  **میتوکندری و هسته چه نقشی دارند؟** میتوکندری مسئول تولید انرژی (ATP) در سلول ها است و هسته حاوی DNA سلول است.
3.  **لیزوزوم ها چه نقشی دارند؟** لیزوزوم ها مسئول تجزیه مواد زائد و مواد زائد سلولی هستند.
4.  **نتیجه گیری:** با توجه به اینکه گلیکولیز در سیتوپلاسم رخ می دهد، آنزیم های گلیکولیز در سیتوپلاسم قرار دارند.

بنابراین، پاسخ صحیح گزینه C است.

**پاسخ:** [C] سیتوپلاسم


---